# Phase S1 Minimal: 2 runs, 50 epochs

In [ ]:
import os, json, math, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch import linalg
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

os.chdir('ThermoRG-NN')
print(f"CWD: {os.getcwd()}")

In [ ]:
# ONLY 2 configs: None vs BN
CONFIGS = [
    ('None',  'none',       False),
    ('BN',    'batchnorm',  False),
]

# ONLY 1 D value
D_VALUES = [64]  # just one
SEEDS = [42]  # just one
EPOCHS = 50  # reduced
BATCH_SIZE = 128
LR = 0.01

print(f"Runs: {len(CONFIGS) * len(D_VALUES) * len(SEEDS)} total")

In [ ]:
# Minimal network
class Net(nn.Module):
    def __init__(self, base_ch=64, norm='none', skip=False):
        super().__init__()
        self.b1 = nn.Conv2d(3, base_ch, 3, padding=1, bias=False)
        self.b2 = nn.Conv2d(base_ch, base_ch*2, 3, padding=1, bias=False)
        self.b3 = nn.Conv2d(base_ch*2, base_ch*2, 3, padding=1, bias=False)
        if norm == 'layernorm':
            self.n1 = nn.LayerNorm([base_ch, 32, 32])
            self.n2 = nn.LayerNorm([base_ch*2, 32, 32])
            self.n3 = nn.LayerNorm([base_ch*2, 32, 32])
        elif norm == 'batchnorm':
            self.n1 = nn.BatchNorm2d(base_ch)
            self.n2 = nn.BatchNorm2d(base_ch*2)
            self.n3 = nn.BatchNorm2d(base_ch*2)
        else:
            self.n1 = self.n2 = self.n3 = nn.Identity()
        self.act = nn.GELU()
        if skip:
            self.s1 = nn.Identity()
            self.s2 = nn.Conv2d(base_ch, base_ch*2, 1) if base_ch != base_ch*2 else nn.Identity()
        else:
            self.s1 = self.s2 = None
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(base_ch*2, 10)

    def forward(self, x):
        x = self.act(self.n1(self.b1(x)))
        if self.s1: x = x + self.s1(x)
        x = self.act(self.n2(self.b2(x)))
        if self.s2: x = x + self.s2(x)
        x = self.act(self.n3(self.b3(x)))
        return self.fc(self.pool(x).squeeze())

print("Network defined.")

In [ ]:
# J_topo
def J_topo(net):
    ws = [m.weight.data for m in net.modules() if isinstance(m, nn.Conv2d)]
    eta_vals, d = [], 3.0
    for w in ws:
        wf = w.view(w.size(0), -1)
        fro = (wf**2).sum().item()
        S = linalg.svd(wf.to('cpu')).S
        De = fro / (S[0].item()**2 + 1e-12)
        eta_vals.append(De / max(d, 1e-8))
        d = De
    return math.exp(-sum(abs(math.log(max(e,1e-12))) for e in eta_vals) / len(eta_vals))

print("J_topo function defined.")

In [ ]:
# Data
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

train_ds = CIFAR10('./data', True, 
    transform=T.Compose([T.RandomCrop(32,padding=4),T.RandomHorizontalFlip(),T.ToTensor(),
                         T.Normalize([0.4914,0.4822,0.4465],[0.2470,0.2435,0.2616])]),
    download=True)
val_ds = CIFAR10('./data', False, transform=T.ToTensor(), download=False)
train_dl = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(val_ds, BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Data: {len(train_ds)} train, {len(val_ds)} val")

In [ ]:
# Train
def train(net, epochs=50):
    net = net.to(device)
    opt = torch.optim.SGD(net.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.CrossEntropyLoss()
    
    for ep in range(epochs):
        net.train()
        for X, y in train_dl:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            crit(net(X), y).backward()
            opt.step()
        sched.step()
        if ep % 10 == 0: print(f"  epoch {ep}/{epochs}")
    
    net.eval()
    loss = 0
    with torch.no_grad():
        for X, y in val_dl:
            loss += crit(net(X.to(device)), y.to(device)).item() * X.size(0)
    return loss / len(val_ds)

print("Training function defined.")

In [ ]:
# MAIN
from datetime import datetime
results = {'timestamp': datetime.now().isoformat(), 'configs': []}
t0 = time.time()

for name, norm, skip in CONFIGS:
    print(f"\n=== [{name}] norm={norm}, skip={skip} ===")
    cfg = {'name': name, 'norm': norm, 'skip': skip}
    
    net_init = Net(64, norm, skip)
    J = J_topo(net_init)
    cfg['J_topo'] = round(J, 4)
    print(f"J_topo = {J:.4f}")
    
    for ch in D_VALUES:
        print(f"  Training ch={ch}...", flush=True)
        net = Net(ch, norm, skip)
        loss = train(net, EPOCHS)
        cfg[f'ch{ch}'] = {'loss': round(loss, 4)}
        print(f"  loss={loss:.4f}")
    
    results['configs'].append(cfg)

print(f"\nTotal time: {(time.time()-t0)/60:.1f} min")

In [ ]:
# Save
out = Path('./phase_s1_results')
out.mkdir(exist_ok=True)
with open(out / 'phase_s1_minimal.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f"Saved to {out/'phase_s1_minimal.json'}")

In [ ]:
# Summary
print("\n" + "="*50)
print("SUMMARY")
print("="*50)
for c in results['configs']:
    loss = c.get('ch64', {}).get('loss', '-')
    print(f"{c['name']}: J={c['J_topo']:.4f}, loss={loss}")